# Structured text and plain-text records - JavaScript

All 7 JavaScript examples from [docs/text.md](https://platob.github.io/yggdryl/text/), in page order.

Generated by `scripts/build_docs_notebooks.py` from the blocks that
`scripts/check_docs_examples.py` compiles and runs, so every code cell below is
an example that passed. An edit here lives until the next build overwrites it.

The cells are unexecuted and call `require`, so they need a CommonJS
JavaScript kernel such as
[IJavascript](https://github.com/n-riesco/ijavascript), with the package
installed beside the notebook:

```console
npm install yggdryl
```

## Plain-text records

In [ ]:
const assert = require('node:assert/strict')
const fs = require('node:fs')
const os = require('node:os')
const path = require('node:path')
const { IOBase, TextOptions } = require('yggdryl')

const textRoot = fs.mkdtempSync(path.join(os.tmpdir(), 'yggdryl-text-'))
const textSource = path.join(textRoot, 'app.log')
fs.writeFileSync(textSource, '  [INFO] id=7 first  \r\n[WARN] id=9 second\n')

const textOptions = new TextOptions()
textOptions.rowheader = '\\[(?<level>[A-Z]+)\\] id=(?<id>\\d+)'
textOptions.lstrip = '^\\s+'
textOptions.rstrip = '\\s+$'

const textHandle = new IOBase(textSource).intoText(textOptions)
const textRows = [...textHandle.readRecords()]
assert.deepEqual(textRows.map((row) => row.rownum), [1n, 2n])
assert.deepEqual(
  textRows.map((row) => Buffer.from(row.body).toString()),
  ['first', 'second'],
)
assert.deepEqual(textRows.map((row) => row.id), [7n, 9n])

const textTarget = new IOBase(path.join(textRoot, 'copy.txt'))
textTarget.overwriteRecords(
  textRows.map((row) => ({ body: row.body })),
  new TextOptions(),
)
assert.equal(textTarget.readBytes().toString(), 'first\nsecond\n')

fs.rmSync(textRoot, { recursive: true, force: true })

## Raw shared-Scalar access

In [ ]:
const assert = require('node:assert/strict')
const { Scalar, json } = require('yggdryl')

const quote = json.loads('{"symbol":"AAPL","price":12.5}', { scalar: true })

assert.ok(quote instanceof Scalar)
assert.equal(quote.get('symbol').asUtf8(), 'AAPL')
assert.equal(quote.path('price').kind, 'f64')
assert.equal(quote.set('venue', 'XNAS').get('venue').asUtf8(), 'XNAS')
assert.deepEqual(quote.asJs(), { price: 12.5, symbol: 'AAPL' })

### Typed `Scalar` families

In [ ]:
const assert = require('node:assert/strict')
const { Scalar } = require('yggdryl')

assert.equal(Scalar.fromJs(40).add(2).asJs(), 42)
assert.ok(Scalar.decimal(1n).divide(Scalar.decimal(2n)).equals(Scalar.decimal(5n, 1)))

## Field-directed parsing

In [ ]:
const assert = require('node:assert/strict')
const { Field, json } = require('yggdryl')

const amount = new Field('amount', 'decimal128(8, 2)', false)
const value = json.loads('"12.50"', { field: amount, scalar: true })

assert.equal(value.kind, 'd128')
assert.equal(value.unscaled, 1250n)
assert.equal(value.scale, 2)

## Raw document codecs

In [ ]:
const assert = require('node:assert/strict')
const { Scalar, codec } = require('yggdryl')

const value = codec.from('{"id":1}', { scalar: true })

assert.ok(value instanceof Scalar)
assert.equal(value.get('id').kind, 'u64')
assert.deepEqual(codec.into(value, { format: 'json' }), Buffer.from('{"id":1}'))

## Formatting

In [ ]:
const assert = require('node:assert/strict')
const { json } = require('yggdryl')

const pretty = json.dumps({ id: 1 }, { indent: 2 })
const compact = json.dumps({ id: 1 }, { indent: null })

assert.deepEqual(pretty, Buffer.from('{\n  "id": 1\n}'))
assert.deepEqual(compact, Buffer.from('{"id":1}'))

## Placeholders

In [ ]:
const assert = require('node:assert/strict')
const { yaml } = require('yggdryl')

const document = 'host: "{{ HOST }}"\nport: "{{ PORT | default(8080) }}"\n'
const value = yaml.loads(document, {
  placeholders: { HOST: 'db.internal' },
})

assert.deepEqual(value, { host: 'db.internal', port: 8080 })